# Equations of motion from a Lagrangian by autograd

*mechanics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/lagrangian_autograd)

Type in only the Lagrangian $L = T - V$ and let torch.autograd derive the Euler–Lagrange equations: the Hessian $\partial^2L/\partial\dot q_i\partial\dot q_j$ is the mass matrix and one linear solve gives $\ddot q$. Applied to a spring (elastic) pendulum.

**Vector form**

$$\frac{d}{dt}\frac{\partial L}{\partial \dot{\mathbf q}} = \frac{\partial L}{\partial \mathbf q},\qquad L = \tfrac12 m(\dot r^2 + r^2\dot\theta^2) + mgr\cos\theta - \tfrac12 k(r-\ell_0)^2$$

**Index (tensor) form**

$$\ddot q_i = H^{-1}_{ij}\Big(\frac{\partial L}{\partial q_j} - C_{jk}\,\dot q_k\Big),\qquad H_{ij} = \frac{\partial^2 L}{\partial\dot q_i\,\partial\dot q_j},\quad C_{ij} = \frac{\partial^2 L}{\partial\dot q_i\,\partial q_j}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $q_i$ | `y[0]` | (2,) | generalised coordinates $(r, \theta)$: spring length and angle |
| $\dot q_i$ | `y[1]` | (2,) | generalised velocities |
| $L(q, \dot q)$ | `spring_pendulum(m, k, l0, g)` | () | the only physics typed in by hand |
| $\partial L/\partial q_i,\ p_i$ | `autograd.grad(L, (q, q_dot), create_graph=True)` | (2,), (2,) | first derivatives; $p_i = \partial L/\partial\dot q_i$ is the canonical momentum |
| $H_{ij},\ C_{ij}$ | `autograd.grad(p[i], (q_dot, q))` | (2, 2) each | row $i$ of the mass matrix and of the mixed second derivatives |
| $C_{jk}\dot q_k$ | `einsum('jk,k->j', C, q_dot)` | (2,) | repeated $k$ ⇒ summed |
| $\ddot q_i$ | `torch.linalg.solve(H, dL_dq - C q_dot)` | (2,) | Euler–Lagrange equations solved for the accelerations |
| $E = \dot q_i p_i - L$ | `energy(L, q, q_dot)` | (T,) | energy function, also by autograd; conserved |

Nothing about the spring pendulum's equations was written by hand: change the Lagrangian and the dynamics follow. Here $H = \operatorname{diag}(m, mr^2)$ depends on $r$, and $C$ contains the Coriolis-like coupling $2mr\dot\theta$. With the default $k \approx 3mg/\ell_0$ the spring frequency is about twice the swing frequency (2:1 resonance), so energy sloshes between bouncing and swinging. The same code works for any number of coordinates; the cost is a few extra backward passes per step.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/lagrangian_autograd/main.py
"""Main loop: parameters → Lagrangian → autograd equations of motion → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    m, k, rest, g = params["m"], params["k"], params["rest_length"], 9.81
    L = equation.spring_pendulum(m, k, rest, g)
    y0 = tensors.initial_state(params["r0"], params["theta0"], dev)

    sol = solve.solve(y0, L, params["t_max"])

    E = sol["E"]
    hanging = rest + m * g / k  # equilibrium length of the hanging spring
    print(f"bounce frequency √(k/m) = {(k / m) ** 0.5:.3f} rad/s")
    print(f"swing frequency √(g/r_eq) ≈ {(g / hanging) ** 0.5:.3f} rad/s")
    print(f"energy: start {E[0]:.5f} J, relative drift {((E - E[0]) / E[0].abs()).abs().max():.1e}")
    return plot.animate(sol, rest, out_dir / "lagrangian_autograd.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/lagrangian_autograd/equation.py
r"""Physics → tensor translation.

Give PyTorch a Lagrangian L(q, \dot q) = T − V and let automatic differentiation derive the equations of motion.
The Euler–Lagrange equations

    \frac{d}{dt}\frac{\partial L}{\partial \dot q_i} = \frac{\partial L}{\partial q_i}

expand by the chain rule (repeated j ⇒ summed) into

    H_{ij} \ddot q_j + C_{ij} \dot q_j = \frac{\partial L}{\partial q_i},
    \qquad H_{ij} = \frac{\partial^2 L}{\partial \dot q_i \partial \dot q_j},
    \qquad C_{ij} = \frac{\partial^2 L}{\partial \dot q_i \partial q_j}

so the acceleration is a linear solve:

    \ddot q_i = H^{-1}_{ij} \left( \frac{\partial L}{\partial q_j} - C_{jk} \dot q_k \right)

H is the (generalised) mass matrix. torch.autograd.grad gives every derivative: one call for ∂L/∂q and
p_i = ∂L/∂\dot q_i (keeping the graph), then one more per component p_i for row i of H and of C.
The conserved energy is

    E = \dot q_i \frac{\partial L}{\partial \dot q_i} - L .

Example: a spring (elastic) pendulum with q = (r, θ), mass m, spring k, natural length ℓ_0:

    L = \tfrac12 m (\dot r^2 + r^2 \dot\theta^2) + m g r \cos\theta - \tfrac12 k (r - \ell_0)^2
"""

from typing import Callable

import torch

Lagrangian = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]  # L(q (..., n), q_dot (..., n)) → (...)


def euler_lagrange(L: Lagrangian, q: torch.Tensor, q_dot: torch.Tensor) -> torch.Tensor:
    r"""\ddot q_i = H^{-1}_{ij}(∂L/∂q_j − C_{jk} \dot q_k), all derivatives by autograd.   (n,), (n,) → (n,)"""
    q = q.detach().requires_grad_(True)
    q_dot = q_dot.detach().requires_grad_(True)
    lagrangian = L(q, q_dot)  # scalar
    # first derivatives; create_graph=True so that we can differentiate them once more.
    # materialize_grads=True returns zeros (instead of an error) when L does not depend on a variable.
    dL_dq, p = torch.autograd.grad(lagrangian, (q, q_dot), create_graph=True, materialize_grads=True)  # (n,) each
    # second derivatives: row i of H and C is the gradient of p_i = ∂L/∂q̇_i
    rows = [torch.autograd.grad(p[i], (q_dot, q), retain_graph=True, materialize_grads=True) for i in range(len(p))]
    H = torch.stack([d_dq_dot for d_dq_dot, _ in rows])  # (n, n)  ∂²L/∂q̇_i∂q̇_j
    C = torch.stack([d_dq for _, d_dq in rows])  # (n, n)  ∂²L/∂q̇_i∂q_j
    q_ddot = torch.linalg.solve(H, dL_dq - torch.einsum("jk,k->j", C, q_dot))
    return q_ddot.detach()


def rhs(y: torch.Tensor, L: Lagrangian) -> torch.Tensor:
    """dy/dt for the state y of shape (2, n): y[0] = q, y[1] = q̇."""
    q, q_dot = y[0], y[1]
    return torch.stack([q_dot, euler_lagrange(L, q, q_dot)])


def energy(L: Lagrangian, q: torch.Tensor, q_dot: torch.Tensor) -> torch.Tensor:
    r"""E = \dot q_i ∂L/∂\dot q_i − L (i summed), by autograd, for many states at once.   (T, n), (T, n) → (T,)"""
    q_dot = q_dot.detach().requires_grad_(True)
    lagrangian = L(q.detach(), q_dot)  # (T,)
    (p,) = torch.autograd.grad(lagrangian.sum(), q_dot)  # (T, n) p_i = ∂L/∂q̇_i (states are independent)
    return (torch.einsum("ti,ti->t", q_dot, p) - lagrangian).detach()


def spring_pendulum(m: float, k: float, rest_length: float, g: float) -> Lagrangian:
    """L(q, q̇) for a mass on a spring swinging in a vertical plane; q = (r, θ), θ from the downward vertical."""

    def L(q: torch.Tensor, q_dot: torch.Tensor) -> torch.Tensor:
        r, theta = q[..., 0], q[..., 1]
        r_dot, theta_dot = q_dot[..., 0], q_dot[..., 1]
        kinetic = 0.5 * m * (r_dot**2 + r**2 * theta_dot**2)
        potential = -m * g * r * torch.cos(theta) + 0.5 * k * (r - rest_length) ** 2
        return kinetic - potential

    return L


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/lagrangian_autograd/tensors.py
"""Build the concrete tensors: the initial generalised coordinates and velocities.

float64 so that energy conservation is limited by the integrator, not by rounding.
"""

import math

import torch

DTYPE = torch.float64


def initial_state(r0: float, theta0_deg: float, device: torch.device | None = None) -> torch.Tensor:
    """y0 of shape (2, 2): released from rest.

    y0[0] = (r, θ)         generalised coordinates q_i
    y0[1] = (0, 0)         generalised velocities q̇_i
    """
    q0 = torch.tensor([r0, math.radians(theta0_deg)], dtype=DTYPE, device=device)  # (2,)
    return torch.stack([q0, torch.zeros_like(q0)])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/lagrangian_autograd/solve.py
"""Integrate the autograd-derived equations of motion with 4th-order Runge–Kutta."""

import torch

from problems._common.integrators import integrate

from .equation import Lagrangian, energy, rhs


def solve(y0: torch.Tensor, L: Lagrangian, t_max: float, dt: float = 0.01) -> dict[str, torch.Tensor]:
    """Return times (T,), coordinates q (T, n), velocities q̇ (T, n) and the energy E (T,).

    Every RK4 stage asks autograd for ∂L/∂q, H and C at the current state: no equation was typed in by hand.
    """
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, L), y0, dt, n_steps)  # (T, 2, n)
    q, q_dot = traj[:, 0], traj[:, 1]
    E = energy(L, q, q_dot)  # (T,) energy at every time, in one batched call
    times = torch.arange(n_steps + 1, device=y0.device, dtype=y0.dtype) * dt
    return {"times": times, "q": q, "q_dot": q_dot, "E": E}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/lagrangian_autograd/plot.py
"""Animate the spring pendulum with a trail, next to r(t) and θ(t)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif


def spring_points(tip: np.ndarray, n_coils: int = 10, width: float = 0.06) -> tuple[np.ndarray, np.ndarray]:
    """A zigzag from the pivot (0, 0) to tip = (x, y), drawn across the line joining them."""
    s = np.linspace(0, 1, 2 * n_coils + 3)
    offset = np.zeros_like(s)
    offset[1:-1:2], offset[2:-1:2] = width, -width
    offset[-2] = 0.0
    side = np.array([-tip[1], tip[0]]) / np.linalg.norm(tip)  # unit vector across the spring
    return s * tip[0] + offset * side[0], s * tip[1] + offset * side[1]


def animate(sol: dict, rest_length: float, path: Path, trail: float = 0.25) -> Path:
    """trail = fraction of the run that the bob's tail remembers."""
    t = sol["times"].cpu().numpy()  # (T,)
    r, theta = sol["q"][:, 0].cpu().numpy(), sol["q"][:, 1].cpu().numpy()  # (T,) each
    bob = np.stack([r * np.sin(theta), -r * np.cos(theta)], axis=-1)  # (T, 2) Cartesian position
    E = sol["E"].cpu().numpy()
    tail = max(2, int(trail * len(t)))
    reach = 1.1 * np.abs(bob).max()

    fig = plt.figure(figsize=(7, 4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1, 1.2])
    ax_swing = fig.add_subplot(grid[:, 0])
    ax_r = fig.add_subplot(grid[0, 1])
    ax_theta = fig.add_subplot(grid[1, 1], sharex=ax_r)

    # --- left: the pendulum ---
    ax_swing.set_xlim(-reach, reach)
    ax_swing.set_ylim(-reach, 0.25 * reach)
    ax_swing.set_aspect("equal", adjustable="box")
    ax_swing.set_xticks([])
    ax_swing.set_yticks([])
    ax_swing.plot([0], [0], "k^", ms=6)
    path_line = ax_swing.plot([], [], lw=1, color=ACCENT[0], alpha=0.5)[0]
    spring = ax_swing.plot([], [], lw=1.2, color="#6b7280")[0]
    dot = ax_swing.plot([], [], "o", ms=10, color=ACCENT[1])[0]
    clock = ax_swing.text(0.03, 0.97, "", transform=ax_swing.transAxes, va="top", fontsize=9)
    ax_swing.set_title(f"E = {E[0]:.3f} J (drift {np.abs(E - E[0]).max():.0e})", fontsize=8)

    # --- right: the two generalised coordinates ---
    ax_r.set_xlim(t[0], t[-1])
    ax_r.set_ylim(r.min() - 0.05, r.max() + 0.05)
    ax_r.axhline(rest_length, color="#9ca3af", lw=0.8, ls="--")
    ax_r.set_ylabel("r (m)")
    ax_theta.set_ylim(np.degrees(theta).min() - 5, np.degrees(theta).max() + 5)
    ax_theta.set_ylabel("θ (°)")
    ax_theta.set_xlabel("t (s)")
    plt.setp(ax_r.get_xticklabels(), visible=False)
    r_line = ax_r.plot([], [], lw=1.3, color=ACCENT[2])[0]
    theta_line = ax_theta.plot([], [], lw=1.3, color=ACCENT[3])[0]

    def update(step: int):
        start = max(0, step - tail)
        path_line.set_data(bob[start : step + 1, 0], bob[start : step + 1, 1])
        spring.set_data(*spring_points(bob[step]))
        dot.set_data([bob[step, 0]], [bob[step, 1]])
        r_line.set_data(t[: step + 1], r[: step + 1])
        theta_line.set_data(t[: step + 1], np.degrees(theta[: step + 1]))
        clock.set_text(f"t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 120), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'm': 1.0,
    'k': 30.0,
    'rest_length': 1.0,
    'r0': 1.4,
    'theta0': 30.0,
    't_max': 15.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.lagrangian_autograd")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.lagrangian_autograd.main")
problem_dir = Path("problems/mechanics/lagrangian_autograd")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
